# 00c. 거시 지표 미래값 추정 (Macro Forecast)

## 📋 개요
04단계(Recursive Extension)에서 미래 날짜의 매크로 피처가 `NaN`으로 채워지는 문제를 해결합니다.  
`macro_regime.parquet`의 과거 실측값을 기반으로 미래 영업일의 거시 지표를 추정하여  
`macro_regime_forecast.parquet`로 저장합니다.

## 📊 추정 대상 및 방법

| 지표 | 성격 | 방법 |
|------|------|------|
| `kospi` | 레벨 시계열 | Damped Holt (φ=0.90) |
| `usd_krw` | 레벨 시계열 | Damped Holt (φ=0.85) |
| `vix` | 레벨 + 평균회귀 | Damped Holt (φ=0.85) |
| `us_return_1d` | 수익률 시계열 | Simple Exponential Smoothing |
| `market_regime` | 파생값 | kospi 추정값으로 재계산 (00b단계 동일 로직) |

## 🔄 데이터 흐름
```
[00b단계] data/99_meta/macro_regime.parquet  (과거 실측값)
    ↓
[00c단계] 지수 평활법 (Holt / SES)
    ↓
data/99_meta/macro_regime_forecast.parquet  (미래 추정값, 동일 스키마)
    ↓
[04단계] macro_regime.parquet + macro_regime_forecast.parquet → concat → left join
```

## ⚠️ 주의사항
- `macro_regime.parquet` 원본은 수정하지 않습니다.
- 00b단계 재실행 시 이 파일에 영향을 주지 않습니다.
- 00c단계는 00b단계 실행 후 선행 실행하거나, 04단계 직전에 실행합니다.
- 추정 horizon은 `krx_calendar.csv`의 미래 영업일 수에 자동 연동됩니다.

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing

from src.utils.config import load_config, ProjectPaths

import warnings
warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 경로

In [ ]:
cfg   = load_config()
paths = ProjectPaths.from_config(cfg)

macro_path         = paths.get_macro_parquet()
forecast_out_path  = paths.get_macro_forecast_parquet()
calendar_path      = paths.get_calendar()

# ── Damped Holt 감쇠 파라미터 ──────────────────────────────────────────────
# φ 값이 클수록 추세를 오래 유지, 작을수록 빨리 수렴
# 레벨 시계열(kospi, usd_krw, vix)에만 적용; us_return_1d는 SES 사용
PHI = {
    'kospi'     : 0.91,   # 추세 지속성이 상대적으로 강함
    'usd_krw'   : 0.89,   # 코스피보다 평균회귀 성격 강함
    'vix'       : 0.88,   # 평균회귀 성격 강함
}

print(f"📂 입력: {macro_path}")
print(f"💾 출력: {forecast_out_path}")
print(f"📅 캘린더: {calendar_path}")

## 2️⃣ 데이터 로드 및 미래 날짜 확인

In [ ]:
# ── 과거 실측값 로드 ────────────────────────────────────────────────────────
assert macro_path.exists(), (
    f"❌ {macro_path} 없음. 00b단계(00b_save_macro_data.ipynb)를 먼저 실행하세요."
)
df_hist = pd.read_parquet(macro_path)
df_hist['date'] = pd.to_datetime(df_hist['date'])
df_hist = df_hist.sort_values('date').reset_index(drop=True)

print(f"✅ 과거 데이터 로드: {len(df_hist):,}일 ({df_hist['date'].iloc[0].date()} ~ {df_hist['date'].iloc[-1].date()})")
print(f"   컬럼: {df_hist.columns.tolist()}")

# ── 미래 영업일 산출 ────────────────────────────────────────────────────────
assert calendar_path.exists(), (
    f"❌ {calendar_path} 없음."
)
df_cal = pd.read_csv(calendar_path)
df_cal['date'] = pd.to_datetime(df_cal['date'])

last_hist_date = df_hist['date'].iloc[-1]
forecast_end = pd.to_datetime(cfg['calendar']['forecast_end'])
future_dates = df_cal[
    (df_cal['date'] > last_hist_date) &
    (df_cal['date'] <= forecast_end)
]['date'].sort_values().reset_index(drop=True)
N = len(future_dates)

assert N > 0, (
    f"❌ 추정할 미래 날짜 없음. 캘린더 마지막 날짜: {df_cal['date'].max().date()}, "
    f"실측 마지막 날짜: {last_hist_date.date()}"
)

print(f"\n📅 추정 대상 미래 영업일: {N}일 ({future_dates.iloc[0].date()} ~ {future_dates.iloc[-1].date()})")

## 3️⃣ 거시 지표 추정

In [ ]:
def forecast_damped_holt(series: pd.Series, n: int, phi: float) -> np.ndarray:
    """
    Damped Holt's Exponential Smoothing으로 n스텝 예측.
    α, β는 statsmodels가 MLE로 자동 추정.
    phi(φ)는 감쇠 파라미터: 1에 가까울수록 추세 지속, 작을수록 빠르게 수렴.
    """
    model = ExponentialSmoothing(
        series.values,
        trend='add',
        damped_trend=True,
    ).fit(
        optimized=True,
        damping_trend=phi,
    )
    return model.forecast(n)


def forecast_ses(series: pd.Series, n: int) -> np.ndarray:
    """
    Simple Exponential Smoothing으로 n스텝 예측.
    수익률 등 정상(stationary) 시계열에 사용.
    모든 스텝의 예측값이 동일 (최근 평균으로 수렴).
    """
    model = SimpleExpSmoothing(series.values).fit(optimized=True)
    return model.forecast(n)


print("✅ 추정 함수 정의 완료")

In [ ]:
# ── 각 지표 추정 ────────────────────────────────────────────────────────────
result = {'date': future_dates}

# 1. KOSPI
print("[1/4] kospi     → Damped Holt", end=' ')
result['kospi'] = forecast_damped_holt(df_hist['kospi'], N, PHI['kospi'])
print(f"| 마지막 실측: {df_hist['kospi'].iloc[-1]:,.1f}  →  추정 마지막: {result['kospi'][-1]:,.1f}")

# 2. USD/KRW
print("[2/4] usd_krw   → Damped Holt", end=' ')
result['usd_krw'] = forecast_damped_holt(df_hist['usd_krw'], N, PHI['usd_krw'])
print(f"| 마지막 실측: {df_hist['usd_krw'].iloc[-1]:,.2f}  →  추정 마지막: {result['usd_krw'][-1]:,.2f}")

# 3. VIX
print("[3/4] vix        → Damped Holt", end=' ')
result['vix'] = forecast_damped_holt(df_hist['vix'], N, PHI['vix'])
print(f"| 마지막 실측: {df_hist['vix'].iloc[-1]:.2f}  →  추정 마지막: {result['vix'][-1]:.2f}")

# 4. us_return_1d
print("[4/4] us_return_1d → SES     ", end=' ')
result['us_return_1d'] = forecast_ses(df_hist['us_return_1d'].dropna(), N)
print(f"| 마지막 실측: {df_hist['us_return_1d'].iloc[-1]:.4f}  →  추정값(고정): {result['us_return_1d'][0]:.4f}")

print("\n✅ 지표 추정 완료")

In [ ]:
# ── market_regime 재계산 (00b단계 동일 로직) ─────────────────────────────────
# kospi_ma200: 과거 199일 + 추정값으로 rolling 계산
# 5일 추정에서 ma200은 거의 변하지 않으므로 마지막 실측 ma200을 기준으로 사용

kospi_all = pd.concat([
    df_hist['kospi'],
    pd.Series(result['kospi'])
], ignore_index=True)

kospi_ma200_all = kospi_all.rolling(window=200, min_periods=200).mean()

# 추정 구간 (마지막 N개)
kospi_forecast_vals  = kospi_all.iloc[-N:].values
kospi_ma200_forecast = kospi_ma200_all.iloc[-N:].values

# 변동성: 과거 vol20의 마지막 값 고정 사용 (5일로 변화 미미)
vol20_last   = df_hist['kospi'].pct_change().rolling(20).std().iloc[-1]
vol_med_last = df_hist['kospi'].pct_change().rolling(20).std().rolling(250).median().iloc[-1]

# 00b단계와 동일한 조건
regime_forecast = np.where(
    kospi_forecast_vals > kospi_ma200_forecast,
    1,   # Bull
    np.where(
        (kospi_forecast_vals < kospi_ma200_forecast) & (vol20_last > vol_med_last),
        -1,  # Bear
        0    # Neutral
    )
)
result['market_regime'] = regime_forecast

print(f"✅ market_regime 재계산 완료: {pd.Series(regime_forecast).value_counts().to_dict()}")

## 4️⃣ 저장 및 검증

In [ ]:
# ── DataFrame 구성 및 저장 ──────────────────────────────────────────────────
# 컬럼 순서를 macro_regime.parquet와 동일하게 맞춤
col_order = [c for c in df_hist.columns if c != 'date']

df_forecast = pd.DataFrame(result)

# 실측에 있으나 추정에 없는 컬럼은 NaN으로 채움 (sp500 원시값 등 중간 계산 컬럼)
for col in col_order:
    if col not in df_forecast.columns:
        df_forecast[col] = np.nan

df_forecast = df_forecast[['date'] + col_order]

df_forecast.to_parquet(forecast_out_path, index=False)
forecast_csv_path  = paths.get_macro_forecast_csv()
df_forecast.to_csv(forecast_csv_path, index=False)

print(f"💾 저장 완료 → {forecast_out_path}")
print(f"   행: {len(df_forecast)}일 / 컬럼: {df_forecast.columns.tolist()}")
display(df_forecast)

In [ ]:
# ── 연속성 검증: 실측 마지막값 vs 추정 첫값 ────────────────────────────────
print("=" * 55)
print("[연속성 검증] 실측 마지막값 → 추정 첫값")
print("=" * 55)

for col in ['kospi', 'usd_krw', 'vix', 'us_return_1d']:
    if col not in df_hist.columns or col not in df_forecast.columns:
        continue
    last_real = df_hist[col].iloc[-1]
    first_fc  = df_forecast[col].iloc[0]
    chg       = (first_fc - last_real) / abs(last_real) * 100 if last_real != 0 else float('nan')
    print(f"  {col:<15}: {last_real:>12.4f}  →  {first_fc:>12.4f}  ({chg:+.2f}%)")

print("=" * 55)
print("✅ 00c단계 완료. 04단계 실행 전 이 파일이 존재해야 합니다.")
print(f"   → {forecast_out_path}")

## 🏁 완료 및 다음 단계

### ✅ 생성된 산출물
- `data/99_meta/macro_regime_forecast.parquet` — 미래 거시 지표 추정값

### 📌 04단계 수정 가이드

`04_forecast_future.ipynb`에서 macro 데이터 로딩 직후 아래 3줄을 추가합니다.

```python
# 기존 코드
df_macro = pd.read_parquet(macro_filepath)

# ✨ 추가: 미래 추정값 병합
forecast_macro_path = meta_dir / 'macro_regime_forecast.parquet'
if forecast_macro_path.exists():
    df_macro_fc = pd.read_parquet(forecast_macro_path)
    df_macro = pd.concat([df_macro, df_macro_fc], ignore_index=True).drop_duplicates('date')
```

### 🔁 실행 순서
```
00b_save_macro_data.ipynb  →  00c_forecast_macro.ipynb  →  01 ~ 05 단계
```

---
**Pipeline Step**: 00c (Macro Forecasting)  
**Method**: Damped Holt (kospi / usd_krw / vix), SES (us_return_1d)  
**Output**: `data/99_meta/macro_regime_forecast.parquet`